# LangChain RAG quality regression testing in CI

This notebook shows how to wire LangChain RAG evals into pytest with snapshot-based regression detection that fails CI when scores drop. The pattern works in any CI pipeline — no SaaS account required.

We use [evalcheck](https://github.com/Boiga7/evalcheck) — an MIT-licensed pytest plugin — to handle the snapshot diff. You could swap in any pytest-shaped eval tool; the *pattern* is what matters.

## What we'll build

1. A tiny RAG chain over two documents.
2. A pytest test for each question that scores faithfulness and correctness.
3. A baseline snapshot that fails CI when future runs drop more than 0.05 below it.

## Setup

In [ ]:
%pip install --quiet langchain langchain-openai langchain-core evalcheck

## A minimal RAG chain

Two documents, one prompt template, one model. Real RAG pipelines have retrievers and indexes; we hard-code the lookup so the example stays focused on the eval part.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

DOCS = {
    "france_capital": "Paris is the capital of France, located on the Seine. The Eiffel Tower stands in the 7th arrondissement.",
    "py_creator": "Guido van Rossum created Python in 1991, while working at CWI in the Netherlands.",
}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the provided context.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)
chain = prompt | llm

def answer(question: str, doc_id: str) -> str:
    return chain.invoke({"context": DOCS[doc_id], "question": question}).content

## Pytest tests with regression tracking

These would normally live in `tests/test_rag.py`. They use `@eval` from evalcheck to wrap a normal pytest test. The decorator scores the return value with the supplied metric, asserts a hard threshold, and (after the first run is blessed as a baseline) asserts no regression beyond a tolerance.

Because evalcheck is a real pytest plugin, plain `pytest` runs everything — no special command, no flags.

In [ ]:
%%writefile test_rag.py
from evalcheck import EvalOutput, eval, faithfulness, correctness

from __main__ import answer, DOCS

@eval(faithfulness, threshold=0.7)
def test_france_grounded_in_context():
    answer_text = answer("What is the capital of France?", "france_capital")
    return EvalOutput(output=answer_text, context=DOCS["france_capital"])

@eval(correctness, threshold=0.7)
def test_python_creator_correct():
    answer_text = answer("Who created Python?", "py_creator")
    return EvalOutput(output=answer_text, expected="Guido van Rossum")

## Run them

Plain `pytest`. No new flags, no new runner. The first run writes results to `.evalcheck/results.json` automatically.

In [ ]:
!pytest test_rag.py -v

## Bless the first run as the baseline

After you've reviewed the first run's scores, copy `.evalcheck/results.json` to `.evalcheck/snapshots/baseline.json` and commit it.

The CLI does it for you:

In [ ]:
!evalcheck snapshot --update

Now any future run that scores more than 0.05 below the baseline will fail the test, surfacing as a regression in CI.

## Making CI fail on regression

Add a step to your GitHub Actions workflow that uploads `.evalcheck/results.json` as an artifact:

```yaml
- name: Run tests
  run: pytest tests/

- name: Upload eval results
  uses: actions/upload-artifact@v4
  if: always()
  with:
    name: evalcheck-results
    path: .evalcheck/results.json
```

Install the [evalcheck GitHub App](https://github.com/apps/evalcheck) on your repo. Now every PR gets a comment showing which evals regressed, improved, or are new — diffed against `baseline.json` on the base branch.

## Where this fits relative to LangChain's own evaluation tools

LangChain's [evaluation framework](https://python.langchain.com/docs/guides/evaluation/) is excellent for in-notebook scoring with rich evaluators (`FaithfulnessEvaluator`, `CriteriaEvaluator`, etc.). evalcheck slots in as the CI integration layer underneath. You could even wrap a LangChain evaluator as a custom evalcheck metric:

```python
from langchain.evaluation import load_evaluator
from evalcheck import EvalOutput, eval

criteria_eval = load_evaluator("criteria", criteria="helpfulness")

def helpfulness(output: str, input: str) -> float:
    result = criteria_eval.evaluate_strings(prediction=output, input=input)
    return float(result["score"])

@eval(helpfulness, threshold=0.7)
def test_response_helpful():
    return EvalOutput(output=answer(...), input="What is...")
```

## Where to go next

- evalcheck on GitHub: https://github.com/Boiga7/evalcheck
- Comparisons vs deepeval, pytest-evals, promptfoo, braintrust: in the repo under `docs/comparisons/`